Either run `source ...` before opening the notebook, or define a variable for the GROMACS bin:

```gm_bin = "<path-to-gmx-bin>"```

and pass it to the `topology_compiler` functions.

**NB** For this example, you may want to use the GROMACS fork with the flow binning code.

Import utility libraries and set the variable for the work directory.

**NB** run the `workdir` cell only once!

In [ ]:
import sys  
sys.path.insert(1, '../libraries/')

In [ ]:
from droplet_spreading import *

In [ ]:
workdir = os.getcwd()
print("Work directory:",workdir)

In [ ]:
# Calibrating the initial droplet dimensions.
# Assuming a contact angle of 10 degrees, the fully spread droplet needs
# not to touch the periodic boundaries.

lx_0 = 5.26800
nbox_x = 10
R0 = 0.5*(nbox_x*lx_0)-5.0
theta_0 = np.deg2rad(10)
r0 = FUN_RADIUS(theta_0,R0)
print("r0 =",r0,"nm")

In [ ]:
# Extending the solid substrate slab to create the surface
extend_substrate("zirconia.gro",nbox_x)

In [ ]:
!ls

In [ ]:
!vmd zirconia-ext.gro

In [ ]:
# Solvating an empty box, from which the droplet will be carved.
# NB if the box is large, the number of atoms can be huge!
# Consider solvating a smaller box, and then resize it to match the length of the surface slab.
solvate_empty_box("box-HFO-1234zeE.gro", "empty.gro", "solvated.gro")

In [ ]:
!ls

In [ ]:
!vmd solvated.gro

In [ ]:
# Defining a lambda to 'carve' a cylinder of liquid out of the solvated box

cx = 26.34
cz = 7.27545
carve_condition = lambda x, y, z : ((x-cx)*(x-cx)+(z-cz)*(z-cz))<=(r0*r0)

In [ ]:
# Carving a mask according to the condisions defined above

# The number of atoms per molecules cannot be inferred, so it needs to be passed as input.
# TODO: create a wrapper that stores this type of information
n_atom_per_mol = 9

carve_gro("solvated.gro",n_atom_per_mol,carve_condition)

In [ ]:
!ls

In [ ]:
!vmd solvated-carved.gro

### Manipulation

Here are a few steps that still need to be implemented in the Python wrapper...

Using `cat`, one can quickly merge the droplet to the substrate, but it's a bit boring since a few lines needs to be removed (e.g. with `vim`) before merging and the number of atoms need to be updated after merging.

Using `gmx insert-molecule` one can also saturate the droplet+substrate box with vapour. This should ease equilibration, but then vapour molecules close to the $z$ periodic boundaries needs to be "carved away":

In [ ]:
zupp = 12.5
zlow = 1.5
carve_condition = lambda x, y, z : (z>zlow)*(z<zupp)

# The input files comes from a NVT simulation of the droplet in its vapour.
# Carving makes space for the surface and the reflective walls at the periodic edges.
carve_gro("md-droplet/nvt.gro",n_atom_per_mol,carve_condition,output_file="carved-temp.gro")

In [ ]:
!vmd carved-temp.gro